In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, TensorDataset, DataLoader
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchmetrics import Accuracy

In [3]:
# Check whether PyTorch see the GPU
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
GPU: NVIDIA GeForce RTX 5070


In [ ]:
# Set the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Using:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

In [5]:
# defining the class to convert to numpy and get and know the length of the dataset
class phishDataset(Dataset):
    def __init__(self, X, y):
        super().__init__()
        # convert to numpy
        self.X = torch.tensor(X.to_numpy(), dtype= torch.float32)
        self.y = torch.tensor(y.to_numpy().squeeze(), dtype=torch.float32) 
        # we want the labels to be in float so we give this else we also give ling which is an ineger will be useing in crossentropy

    def __len__(self):
        return self.X.shape[0]

    def __getitem__(self, idx):
        features = self.X[idx]
        label = self.y[idx]
        return features, label

In [6]:
X = pd.read_csv("../../data/Processed/phishing_websites_dataset/X.csv")
y = pd.read_csv("../../data/Processed/phishing_websites_dataset/y.csv")
print(y.value_counts())
y = y.replace({-1: 0, 1: 1})
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.33,
    random_state=42
)

result
 1        6157
-1        4898
Name: count, dtype: int64


In [7]:
# converting the labels to 0 to 1
y.value_counts()

result
1         6157
0         4898
Name: count, dtype: int64

In [8]:
# we import thr dataset and call it in the DataLoader
dataset_train = phishDataset(X_train, y_train)
dataset_test = phishDataset(X_test, y_test)

In [9]:
data_loader_train = DataLoader(
    dataset_train,
    batch_size = 32,
    shuffle = True
)
data_loader_test = DataLoader(
    dataset_test,
    batch_size=32,
    shuffle=False
)

In [10]:
# looking into in datapoint
dataset_train.__getitem__(222)

(tensor([-1., -1.,  1.,  1.,  1.,  1.,  1.,  1., -1.,  1.,  1.,  1.,  1.,  0.,
          1., -1.,  1.,  1.,  0.,  1.,  1.,  1.,  1.,  1.,  1., -1., -1.,  1.,
          1.,  1.]),
 tensor(1.))

In [11]:
# To check whether your Dataset and DataLoader are producing the data in the shape and 
# format your neural network expects.
features_train, label_train = next(iter(data_loader_train))
print(f"Features : {features_train}, \nLabels: {label_train}")
print("Features_train shape:", features_train.shape)
print("Labels_train shape:", label_train.shape)

Features : tensor([[ 1., -1.,  1.,  1.,  1., -1.,  1.,  1., -1.,  1.,  1.,  1.,  1.,  1.,
          1., -1.,  1.,  1.,  0.,  1.,  1.,  1.,  1.,  1., -1.,  1.,  1., -1.,
          1.,  1.],
        [-1., -1., -1.,  1., -1.,  1.,  1.,  1., -1.,  1.,  1., -1.,  1.,  1.,
         -1., -1.,  1., -1.,  1.,  1.,  1.,  1.,  1.,  1., -1., -1., -1.,  1.,
          1.,  1.],
        [ 1., -1.,  1.,  1.,  1., -1.,  1., -1.,  1.,  1.,  1.,  1.,  1.,  0.,
         -1., -1.,  1.,  1.,  0.,  1.,  1.,  1.,  1., -1., -1.,  0., -1.,  1.,
          1.,  1.],
        [ 1.,  0.,  1.,  1.,  1., -1.,  0., -1.,  1.,  1.,  1.,  1., -1., -1.,
          1.,  1., -1.,  1.,  0.,  1.,  1.,  1.,  1.,  1.,  1.,  1., -1., -1.,
          0.,  1.],
        [ 1.,  1.,  1.,  1.,  1., -1.,  0., -1., -1., -1., -1.,  1.,  1.,  1.,
         -1.,  1., -1.,  1.,  0., -1.,  1., -1., -1., -1.,  1.,  1., -1.,  1.,
          1., -1.],
        [-1., -1.,  1.,  1.,  1., -1.,  1.,  1., -1.,  1.,  1.,  1.,  1.,  0.,
          0., -1.,  

# The OOP approach to define the model architecture
|                              | `nn.Sequential` | Class-based `nn.Module` |
| ---------------------------- | --------------- | ----------------------- |
| Easy to write                | ⭐⭐⭐⭐⭐           | ⭐⭐⭐                     |
| Simple straight network      | ✅               | ✅                       |
| Custom forward logic         | ❌/limited       | ✅                       |
| Branches                     | ❌               | ✅                       |
| Skip connections             | ❌/awkward       | ✅                       |
| Multiple inputs/outputs      | ❌/awkward       | ✅                       |
| Most beginner tabular models | ✅               | ✅                       |
| Complex architectures        | ❌               | ✅                       |


In [12]:
# define the custom model architecture
class Net1(nn.Module):
    def __init__(self):
        super().__init__()
        # Define the three linear layers
        self.fc1 = nn.Linear(30,64)
        self.fc2 = nn.Linear(64,32)
        self.fc3 = nn.Linear(32,1)
        
    def forward(self, x):
        # Pass x through linear layers adding activations
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = F.sigmoid(self.fc3(x))
        return x

In [45]:
class Net2(nn.Module):
    def __init__(self):
        super().__init__()

        # Input: 30 features
        self.fc1 = nn.Linear(30, 128)
        self.dropout1 = nn.Dropout(p=0.3)

        # Hidden layers
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 32)
        self.fc4 = nn.Linear(32, 16)

        # Output layer
        self.fc5 = nn.Linear(16, 1)

    def forward(self, x):

        x = F.relu(self.fc1(x))
        x = self.dropout1(x)
        x = F.relu(self.fc2(x))
        x = F.relu(self.fc3(x))
        x = F.relu(self.fc4(x))

        # Binary classification output
        x = F.sigmoid(self.fc5(x))

        return x

In [46]:
# Defining the loss function and optimizer
model = Net2().to(device)
criterion = nn.BCELoss()
optimizer = optim.SGD(model.parameters(), lr = 0.01)
num_epochs = 50

In [47]:
# Iterate over epochs and training batches
"""
- Clear Gradients
- Forward pass : get model's outputs
- compute loss
- compute gradients
- optimizer's step: update params
"""


for epoch in range(num_epochs):
    model.train()
    # Track statistics for this epoch
    train_loss = 0.0
    correct = 0
    total = 0
    
    for features, labels in data_loader_train:
        optimizer.zero_grad()

        # movies these to the gpu
        features = features.to(device)
        labels = labels.to(device) 
        
        outputs = model(features)
        loss = criterion(
            outputs, labels.view(-1,1)
        )
        loss.backward()
        optimizer.step()

        train_loss += loss.item() * features.size(0)
        
        # -------------------------
        # Calculate accuracy
        # -------------------------
        predictions = (outputs >= 0.5).float()

        correct += (
            predictions == labels.view(-1, 1)
        ).sum().item()

        total += labels.size(0)

    # Mean loss over entire training dataset
    train_loss /= len(data_loader_train.dataset)

    # Accuracy over entire training dataset
    train_accuracy = correct / total
    
    print(
        f"Epoch {epoch + 1:3d} | "
        f"Train Loss: {train_loss:.4f} | "
        f"Train Accuracy: {train_accuracy:.4f}"
    )
        

        

Epoch   1 | Train Loss: 0.6946 | Train Accuracy: 0.5072
Epoch   2 | Train Loss: 0.6845 | Train Accuracy: 0.5517
Epoch   3 | Train Loss: 0.6752 | Train Accuracy: 0.5501
Epoch   4 | Train Loss: 0.6547 | Train Accuracy: 0.6065
Epoch   5 | Train Loss: 0.5878 | Train Accuracy: 0.7767
Epoch   6 | Train Loss: 0.4146 | Train Accuracy: 0.8762
Epoch   7 | Train Loss: 0.2864 | Train Accuracy: 0.9017
Epoch   8 | Train Loss: 0.2466 | Train Accuracy: 0.9113
Epoch   9 | Train Loss: 0.2294 | Train Accuracy: 0.9155
Epoch  10 | Train Loss: 0.2141 | Train Accuracy: 0.9218
Epoch  11 | Train Loss: 0.2088 | Train Accuracy: 0.9237
Epoch  12 | Train Loss: 0.2022 | Train Accuracy: 0.9274
Epoch  13 | Train Loss: 0.1954 | Train Accuracy: 0.9268
Epoch  14 | Train Loss: 0.1921 | Train Accuracy: 0.9264
Epoch  15 | Train Loss: 0.1908 | Train Accuracy: 0.9283
Epoch  16 | Train Loss: 0.1836 | Train Accuracy: 0.9294
Epoch  17 | Train Loss: 0.1823 | Train Accuracy: 0.9321
Epoch  18 | Train Loss: 0.1842 | Train Accuracy:

In [48]:
# Model Evaluation
# here we are using torchmetrics library
acc = Accuracy(task="binary").to(device) # Set up accuracy metric

model.eval() # Put model in eval mode
with torch.no_grad(): # iterate over test data batches with no gradients
    for features, labels in data_loader_test: 
        # Move test batch to GPU
        features = features.to(device)
        labels = labels.to(device)
        outputs = model(features)
        preds = (outputs >= 0.5).float()
        acc(preds, labels.view(-1,1))

accuracy = acc.compute()
print(f"Accuracy : {accuracy}")
        

Accuracy : 0.9460126161575317


In [49]:
# Model Evaluation mannual calculation
model.eval()
correct = 0
total = 0
with torch.no_grad():
    for features, labels in data_loader_test:
        # Move test batch to GPU
        features = features.to(device)
        labels = labels.to(device)
        # Forward pass
        outputs = model(features)
        # Convert probability → 0 or 1
        preds = (outputs >= 0.5).float()
        # Count correct predictions
        correct += (preds == labels.view(-1, 1)).sum().item()
        # Count total predictions
        total += labels.size(0)
# Calculate accuracy
accuracy = correct / total
print(f"Accuracy: {accuracy:.4f}")

Accuracy: 0.9460


In [ ]:
# Combination of weight intialization and Batch Normalization
import torch.nn.init as init

class Net(nn.Module):
    def __init__(self):
        super().__init__()
        self.fc1 = nn.Linear(9, 16)
        # Add two batch normalization layers
        self.bn1 = nn.BatchNorm1d(16)
        self.fc2 = nn.Linear(16, 8)
        self.bn2 = nn.BatchNorm1d(8)
        self.fc3 = nn.Linear(8, 1)
        
        init.kaiming_uniform_(self.fc1.weight)
        init.kaiming_uniform_(self.fc2.weight)
        init.kaiming_uniform_(self.fc3.weight, nonlinearity="sigmoid")
    
    def forward(self, x):
        x = self.fc1(x)
        x = self.bn1(x)
        x = nn.functional.elu(x)

        # Pass x through the second set of layers
        x = self.fc2(x)
        x = self.bn2(x)
        x = nn.functional.elu(x)

        x = nn.functional.sigmoid(self.fc3(x))
        return x